<a href="https://colab.research.google.com/github/trainocate-japan/openai_api_app/blob/main/chapter2/%E7%AC%AC2%E7%AB%A0%20(%E3%82%AA%E3%83%97%E3%82%B7%E3%83%A7%E3%83%B3)%20%E9%80%A3%E7%B6%9A%E3%81%97%E3%81%9F%E5%AF%BE%E8%A9%B1%E6%A9%9F%E8%83%BD%E3%81%AE%E5%AE%9F%E8%A3%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#アプリケーション内で連続した対話ができる機能を追加する

In [ ]:
# !pip install openai
# streamlit関連パッケージのインストール
!pip install streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!which cloudflared && cloudflared --version

In [ ]:
%%writefile app.py
from openai import OpenAI
import streamlit as st

client = OpenAI(api_key="your api key")

st.title("Hello Streamlit")
user_input = st.text_input("あなたのメッセージを入力してください:")

if st.button("送信"):
    if "history" not in st.session_state:
        st.session_state["history"] = [{"role": "system", "content": "You are a helpful assistant."}]

    st.session_state["history"].append({"role": "user", "content": user_input})

    # OpenAI APIを使用して応答を生成
    response = client.chat.completions.create(
        model= "gpt-4o-mini",
        messages=st.session_state["history"],
        )

    # 応答を対話履歴に追加
    st.session_state["history"].append({"role": "assistant", "content": response.choices[0].message.content})

    # 対話履歴を表示
    for message in st.session_state["history"][:-1]:
        if message["role"] == "user":
            st.text_area("User", message["content"])
        elif message["role"] == "assistant":
            st.text_area("Bot", message["content"])
    st.text_area("Bot", response.choices[0].message.content)

In [ ]:
# streamlitのrunコマンドでapp.pyを立ち上げ、localtunnelを用いてアプリ公開
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.fileWatcherType none \
  > /tmp/st.log 2>&1 &
!for i in {1..60}; do curl -fsS http://localhost:8501/healthz && echo "Streamlit is up" && break || sleep 1; done

# トンネル起動（ログにURLが出る）
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /tmp/cf.log 2>&1 &

# URLがログに出るまで最大60秒待って抽出
!for i in {1..60}; do \
  URL=$(grep -o "https://[0-9a-z.-]*trycloudflare.com" -m 1 /tmp/cf.log); \
  if [ -n "$URL" ]; then echo "PUBLIC URL: $URL"; break; fi; \
  sleep 1; \
done